# Harfi size study: Qwen3-0.6B on v4's data

How small can Harfi's model be? This trains **Qwen3-0.6B** (600 million parameters) with **exactly v4's recipe and
data** (LoRA rank 32, alpha 64, lr 2e-4, 3 epochs, best epoch by validation loss kept; the `harfi-v5` export, 122,492
examples), so the only difference from v4 (Llama 3.2 1B, 808 MB) is the base model.

Qwen3 trains in fp16 on the T4, like Llama. Its thinking mode is turned off in the prompt, as for the Qwen3-1.7B run in October.

Licence: Apache-2.0.

**Output:** `arabizi-qwen3-0.6b-v1-Q4_K_M.gguf` and `arabizi-qwen3-0.6b-v1-Q8_0.gguf`, their SHA-256 files and `run.json` (with the exact prompt the
engine needs).

**How to run:**
1. Add the `harfi-v5` dataset (the same one v4 used): Add Input -> your dataset.
2. Settings -> Accelerator -> **GPU T4** · **Internet on**.
3. If the size-study changes are not merged into `main` yet, set `REF` below to `feat/small-models-kaggle`.
4. **Save Version -> Save & Run All (Commit)**: about 4 to 5 hours in the background (under Kaggle's 12-hour limit).
5. From the version's **Output** tab, **Download all** and send the zip. It is scored on the laptop against v4 on
   the dev sets, the 300-message comparison set and the real chats, with its size and speed.

The Gemma and Qwen notebooks are independent: run them at the same time if your Kaggle quota allows two GPU sessions.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as v4


In [ ]:
REF = "main"  # or "feat/small-models-kaggle" until the size-study changes are merged
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
assert summary["rows_total"] == 122492, "this should be v4's export (harfi-v5): the comparison needs the same data"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# v4's settings (rank 32, alpha 64, lr 2e-4, 3 epochs), only the base model changes.
BASE, TAG, NAME = "qwen3-0.6b", "v1", "arabizi-qwen3-0.6b-v1"

print(f"=== {BASE} {TAG}", flush=True)
# check=True: a failed run stops the notebook instead of exporting nothing.
subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                "--tag", TAG, "--rank", "32", "--alpha", "64"], check=True)
run = max(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", NAME, run, "Q4_K_M Q8_0"], check=True)
subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)


## Results and download

The best validation loss is only a first look (v4's was 0.183); accuracy on real messages, measured on the laptop,
decides. From the version's **Output** tab, use **Download all** (one zip).


In [ ]:
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min, {r['precision']})")
    for s in r["public_samples"]:
        print(f"  {s['arabizi']}\n    -> {s['output']}\n    want {s['reference']}")
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"
